# Notebook 07: Meson Correlators and Masses

**Learning objectives:**
- Understand meson operators ($\pi$, $\sigma$, $\rho$) and their quantum numbers
- Calculate meson two-point correlators $C(t)$
- Extract effective masses $m_{\rm eff}(t) = \ln[C(t)/C(t+1)]$
- Fit the plateau to obtain the ground-state mass
- Study the pion mass vs quark mass (GMOR relation)
- Compare the $\pi$, $\sigma$, $\rho$ mass hierarchy

**Prerequisites:** Notebooks 05-06

**From modular code:** `MesonBase.get_meson_gamma_matrix()`, `PionCalculator.calculate_pion_mass()`,
`SigmaCalculator.calculate_sigma_mass()`, `RhoCalculator.calculate_rho_mass()`,
`MesonIntegration.calculate_meson_spectrum()`

In [ ]:
from notebook_utils import setup_paths, load_config, plot_correlator, plot_effective_mass
REPO = setup_paths()

import os
import numpy as np
import matplotlib.pyplot as plt
import su2
from MesonBase import (get_meson_gamma_matrix,
                       build_wilson_dirac_matrix,
                       generate_identity_gauge_field,
                       create_point_source,
                       solve_dirac_system,
                       calculate_effective_mass,
                       fit_plateau,
                       get_gamma_matrices)
from PionCalculator import calculate_pion_correlator, calculate_pion_mass
from SigmaCalculator import calculate_sigma_correlator, calculate_sigma_mass
from RhoCalculator import calculate_rho_correlator, calculate_rho_mass
from MesonIntegration import calculate_meson_mass, calculate_meson_spectrum

## 1. Meson Operators

Mesons are quark-antiquark bound states. The meson operator
$M_\Gamma(x) = \bar\psi(x)\,\Gamma\,\psi(x)$
creates a meson with quantum numbers determined by $\Gamma$:

| Meson | $\Gamma$ | $J^{PC}$ | Description |
|-------|----------|----------|-------------|
| Pion ($\pi$) | $\gamma_5$ | $0^{-+}$ | Pseudoscalar, Goldstone boson |
| Sigma ($\sigma$) | $\mathbb{1}$ | $0^{++}$ | Scalar, chiral partner of pion |
| Rho ($\rho$) | $\gamma_i$ | $1^{--}$ | Vector meson (3 polarizations) |

In [ ]:
# Examine gamma matrices for each channel
for ch in ['pion', 'sigma', 'rho_x']:
    info = get_meson_gamma_matrix(ch, verbose=False)
    print(f"{ch:8s}: J^PC = {info['JPC']}, Gamma =")
    print(f"  {np.array2string(info['gamma'], precision=1)}\n")

## 2. Computing a Pion Correlator

The meson two-point correlator is the expectation value:
$$C_\Gamma(t) = \sum_{\vec{x}} \langle 0 | \bar\psi(\vec{x},t)\,\Gamma\,\psi(\vec{x},t) \; \bar\psi(0)\,\Gamma\,\psi(0) | 0 \rangle$$

For the pion ($\Gamma = \gamma_5$), Wick contraction gives:
$$C_\pi(t) = -\sum_{\vec{x}} \mathrm{Tr}\bigl[\gamma_5\, S(\vec{x},t;\, 0)\, \gamma_5\, S^\dagger(\vec{x},t;\, 0)\bigr]$$

For large $t$, the correlator decays exponentially:
$$C(t) \xrightarrow{t \gg 1} A\, e^{-m_\pi\, t}$$

so the mass can be extracted from the slope on a log plot.

In [ ]:
La = [4, 4, 4, 4]

# TRY: Change mass from 0.2 to 0.05 — lighter quarks = lighter pion (GMOR)
# TRY: Change wilson_r from 0.5 to 1.0 — the effective mass shifts by 4*delta_r
mass = 0.2
wilson_r = 0.5

# Use identity gauge field for clean signal
U_free, _ = generate_identity_gauge_field(La)

result = calculate_meson_mass(U_free, mass, 'pion', La,
                              wilson_r=wilson_r, verbose=True)

In [ ]:
# Plot correlator and effective mass
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

plot_correlator(result['correlator'], title='Pion correlator (free field)', ax=ax1)
plot_effective_mass(result['correlator'], title='Pion effective mass', ax=ax2)

plt.tight_layout()
plt.show()

print(f"Extracted pion mass: {result['meson_mass']:.4f} +/- {result['meson_error']:.4f}")
print(f"Effective mass = m + 4r = {mass + 4*wilson_r:.4f}")

## 3. Full Spectrum ($\pi$, $\sigma$, $\rho$)

In QCD the expected mass hierarchy is $m_\pi < m_\rho < m_\sigma$.
The pion is light because it's a (pseudo-)Goldstone boson of chiral symmetry.

In [ ]:
channels = ['pion', 'sigma', 'rho_x', 'rho_y', 'rho_z']
results = {}

for ch in channels:
    r = calculate_meson_mass(U_free, mass, ch, La,
                            wilson_r=wilson_r, verbose=False)
    results[ch] = r
    print(f"{ch:8s}: M = {r['meson_mass']:.4f} +/- {r['meson_error']:.4f}")

In [ ]:
# Visualize the spectrum
names = list(results.keys())
masses = [results[ch]['meson_mass'] for ch in names]
errors = [results[ch]['meson_error'] for ch in names]

plt.figure(figsize=(8, 5))
plt.bar(names, masses, yerr=errors, capsize=5, alpha=0.7,
        color=['C0', 'C1', 'C2', 'C2', 'C2'], edgecolor='k')
plt.ylabel('Meson mass (lattice units)', fontsize=13)
plt.title('Meson spectrum (single config, free field)', fontsize=14)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 4. $M_\pi^2$ vs $m_q$: The GMOR Relation

The **Gell-Mann--Oakes--Renner** (GMOR) relation is one of the most important
results connecting chiral symmetry to hadron physics:
$$M_\pi^2 = \frac{2\,m_q\,|\langle\bar\psi\psi\rangle|}{f_\pi^2}$$

The key prediction is linearity: $M_\pi^2 \propto m_q$.
This is a hallmark of the pion being a **pseudo-Goldstone boson** of
spontaneously broken chiral symmetry.
As $m_q \to 0$, the pion becomes massless (the Goldstone limit).

In [ ]:
# TRY: Add more quark masses (e.g. 0.01, 0.02) to probe closer to the chiral limit
# TRY: Very small masses may give noisy/unstable correlators on a small lattice
quark_masses = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4]
pion_masses = []

for mq in quark_masses:
    r = calculate_meson_mass(U_free, mq, 'pion', La,
                            wilson_r=wilson_r, verbose=False)
    pion_masses.append(r['meson_mass'])
    print(f"  m_q = {mq:.2f} -> M_pi = {r['meson_mass']:.4f}")

mq_arr = np.array(quark_masses)
mp_arr = np.array(pion_masses)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# M_pi vs m_q
ax1.plot(mq_arr, mp_arr, 'o-', markersize=8)
ax1.set_xlabel(r'$m_q$ (lattice units)', fontsize=13)
ax1.set_ylabel(r'$M_\pi$', fontsize=13)
ax1.set_title(r'Pion mass vs quark mass')
ax1.grid(True, alpha=0.3)

# M_pi^2 vs m_q (should be linear per GMOR)
ax2.plot(mq_arr, mp_arr**2, 's-', markersize=8, color='C1')
# Linear fit
coeffs = np.polyfit(mq_arr, mp_arr**2, 1)
ax2.plot(mq_arr, np.polyval(coeffs, mq_arr), 'r--',
         label=f'Linear fit: slope = {coeffs[0]:.2f}')
ax2.set_xlabel(r'$m_q$ (lattice units)', fontsize=13)
ax2.set_ylabel(r'$M_\pi^2$', fontsize=13)
ax2.set_title(r'GMOR relation: $M_\pi^2 \propto m_q$')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Interacting Gauge Field

Now let's repeat the pion calculation on a non-trivial gauge background.
This is where the gluon dynamics create binding.

In [ ]:
config_path = os.path.join(REPO, "configs", "sample_4x4x4x4",
                           "random_4x4x4x4.pkl")
U_rand, meta = load_config(config_path)
U_int = [None, U_rand]

result_int = calculate_meson_mass(U_int, 0.2, 'pion', La,
                                  wilson_r=0.5, verbose=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
plot_correlator(result_int['correlator'],
               title='Pion correlator (interacting)', ax=ax1)
plot_effective_mass(result_int['correlator'],
                   title='Pion effective mass (interacting)', ax=ax2)
plt.tight_layout()
plt.show()

## Exercises

1. Compare the pion mass on the free-field and interacting configs.
   Why is the interacting case noisier?
2. Calculate $M_\sigma / M_\pi$ on the free field.
   How does this ratio compare to the real-world value ($\approx 4$)?
3. Compute the rho mass for all 3 polarizations ($\rho_x, \rho_y, \rho_z$).
   Are they degenerate? What breaks rotational symmetry on a single config?
4. Plot $M_\pi, M_\sigma, M_\rho$ as a function of $m_q$.
   Which particle is always lightest?
5. **GMOR quantitative fit**: Using the mass scan data from Section 4,
   fit $M_\pi^2 = c_0 + c_1 \cdot m_q$ with `np.polyfit`.
   The $x$-intercept $m_{\rm crit} = -c_0/c_1$ is where $M_\pi \to 0$.
   For Wilson fermions, $m_{\rm crit} \neq 0$ due to additive renormalization.
   Compare your value to the free-field expectation $m_{\rm crit} = -4r$.
6. **Excited state contamination**: In the effective mass plot from Section 2,
   identify the region where $m_{\rm eff}(t)$ is still decreasing (excited
   states contribute). At what $t$ does the plateau begin?
   Try fitting only $t \geq 2$ vs $t \geq 1$ — how does the extracted
   mass change? A robust extraction requires the plateau to be stable
   under variations of $t_{\min}$.